In [48]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from tabulate import tabulate

In [126]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from tabulate import tabulate

class TrendAnalyzer:
    def __init__(self, asset_id, data_path, btc_data_path, use_btc_adjusted=True):
        self.asset_id = asset_id
        self.data_path = f"{data_path}{asset_id}_candles.csv"
        self.btc_data_path = btc_data_path
        self.use_btc_adjusted = use_btc_adjusted
        self.data = self._load_data()
        self.price_column = self._set_price_column()
        print(f"Price column selected: {self.price_column}")
        self.ma_periods = {
            'Short Term': [3, 5, 7, 14],
            'Medium Term': [21, 30, 45, 63],
            'Long Term': [84, 100, 120, 150, 200, 252, 365]
        }
        self._calculate_indicators()
        self.classified_data = self._create_classified_data()

    def _load_data(self):
        # Load asset data
        data = pd.read_csv(self.data_path)
        data['date'] = pd.to_datetime(data['date'])
        data.dropna(inplace=True)
        print(f"Initial columns after loading asset data: {list(data.columns)}")

        # Load and merge Bitcoin data if not Bitcoin
        if self.asset_id != 'bitcoin':
            btc_data = pd.read_csv(self.btc_data_path)
            btc_data['date'] = pd.to_datetime(btc_data['date'])
            btc_data = btc_data[['date', 'close']].rename(columns={'close': 'btc_close'})
            print(f"Bitcoin data columns: {list(btc_data.columns)}")
            data = data.merge(btc_data, on='date', how='left')
            print(f"Columns after merge: {list(data.columns)}")
            if 'close' in data.columns and 'btc_close' in data.columns:
                data[f'{self.asset_id}_btc'] = data['close'] / data['btc_close']
            else:
                print(f"Missing columns after merge. Available columns: {list(data.columns)}")
            data = data.drop(columns=['btc_close'], errors='ignore')
            print(f"Columns after dropping 'btc_close': {list(data.columns)}")
        return data

    def _set_price_column(self):
        if self.use_btc_adjusted and self.asset_id != 'bitcoin':
            return f'{self.asset_id}_btc'
        else:
            return 'close'

    def _calculate_indicators(self):
        if self.price_column not in self.data.columns:
            raise KeyError(f"Column '{self.price_column}' not found. Available columns: {list(self.data.columns)}")
        self.data[self.price_column] = pd.to_numeric(self.data[self.price_column], errors='coerce')
        self.data.dropna(subset=[self.price_column], inplace=True)
        print(f"Columns after numeric conversion and dropna: {list(self.data.columns)}")
        
        for term, periods in self.ma_periods.items():
            for period in periods:
                self.data[f'SMA{period}'] = self.data[self.price_column].rolling(
                    window=period, min_periods=period
                ).mean()
        
        for term, periods in self.ma_periods.items():
            for period in periods:
                self.data[f'RoC{period}'] = (
                    (self.data[self.price_column] - self.data[self.price_column].shift(period))
                    / self.data[self.price_column].shift(period) * 100
                )
        print(f"Columns after calculating indicators: {list(self.data.columns)}")

    def classify_trend(self, mas, rocs):
        n_mas = len(mas.dropna())
        n_rocs = len(rocs.dropna())
        if n_mas == 0 or n_rocs == 0:
            return "Neutral"
        
        pos_mas = 0
        for col in mas.index:
            period = int(col.replace('SMA', ''))
            current_ma = self.data[f'SMA{period}'].iloc[-1]
            prev_ma = self.data[f'SMA{period}'].iloc[-2] if len(self.data) > 1 else np.nan
            if not np.isnan(current_ma) and not np.isnan(prev_ma) and current_ma > prev_ma:
                pos_mas += 1
        
        pos_mas = pos_mas / n_mas * 100 if n_mas > 0 else 0
        pos_rocs = sum(1 for roc in rocs if roc > 0) / n_rocs * 100 if n_rocs > 0 else 0
        
        avg_pos = (pos_mas + pos_rocs) / 2
        
        if avg_pos >= 75:
            return "Strong Bull"
        elif avg_pos >= 50:
            return "Weak Bull"
        elif avg_pos <= 25:
            return "Strong Bear"
        elif avg_pos <= 40:
            return "Weak Bear"
        else:
            return "Neutral"

    def _create_classified_data(self):
        # Initialize a copy of the original data
        classified_data = self.data.copy()
        
        # Calculate classifications for each day
        short_term_mas = [f'SMA{period}' for period in self.ma_periods['Short Term']]
        medium_term_mas = [f'SMA{period}' for period in self.ma_periods['Medium Term']]
        long_term_mas = [f'SMA{period}' for period in self.ma_periods['Long Term']]
        
        short_term_rocs = [f'RoC{period}' for period in self.ma_periods['Short Term']]
        medium_term_rocs = [f'RoC{period}' for period in self.ma_periods['Medium Term']]
        long_term_rocs = [f'RoC{period}' for period in self.ma_periods['Long Term']]

        classified_data['Short Term'] = classified_data.apply(
            lambda row: self.classify_trend(row[short_term_mas], row[short_term_rocs]), axis=1
        )
        classified_data['Medium Term'] = classified_data.apply(
            lambda row: self.classify_trend(row[medium_term_mas], row[medium_term_rocs]), axis=1
        )
        classified_data['Long Term'] = classified_data.apply(
            lambda row: self.classify_trend(row[long_term_mas], row[long_term_rocs]), axis=1
        )
        
        # Calculate Overall classification for each day
        classified_data['Overall'] = classified_data.apply(
            lambda row: self._compute_overall_classification(
                [row['Short Term'], row['Medium Term'], row['Long Term']]
            ), axis=1
        )
        
        return classified_data

    def _compute_overall_classification(self, trends):
        overall_pos = sum(1 for c in trends if c in ["Strong Bull", "Weak Bull"]) / len(trends) * 100
        if overall_pos >= 75:
            return "Strong Bull"
        elif overall_pos >= 50:
            return "Weak Bull"
        elif overall_pos <= 25:
            return "Strong Bear"
        elif overall_pos <= 40:
            return "Weak Bear"
        else:
            return "Neutral"

    def analyze(self):
        classifications = {}
        for term, periods in self.ma_periods.items():
            mas = self.data[[f'SMA{period}' for period in periods]].iloc[-1]
            rocs = self.data[[f'RoC{period}' for period in periods]].iloc[-1]
            classification = self.classify_trend(mas, rocs)
            classifications[term] = classification
            print(f"{term} classification: {classification}")
        overall_trends = list(classifications.values())
        overall_pos = sum(1 for c in overall_trends if c in ["Strong Bull", "Weak Bull"]) / len(overall_trends) * 100
        if overall_pos >= 75:
            overall_classification = "Strong Bull"
        elif overall_pos >= 50:
            overall_classification = "Weak Bull"
        elif overall_pos <= 25:
            overall_classification = "Strong Bear"
        elif overall_pos <= 40:
            overall_classification = "Weak Bear"
        else:
            overall_classification = "Neutral"
        classifications['Overall'] = overall_classification
        print(f"Overall classification: {overall_classification}")
        return classifications

    def create_table(self):
        classifications = self.analyze()
        table_data = {
            'Ticker': [self.asset_id.upper()],
            'Short Term Trend': [classifications['Short Term']],
            'Medium Term Trend': [classifications['Medium Term']],
            'Long Term Trend': [classifications['Long Term']],
            'Overall Trend': [classifications['Overall']]
        }
        print(f"Table data: {table_data}")
        table = tabulate(table_data, headers='keys', tablefmt='pretty', showindex=False)
        print(table)
        return None

    def create_chart(self):
        selected_smas = [14, 30, 63, 200]
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=self.data['date'],
            y=self.data[self.price_column],
            mode='lines',
            name='Price',
            line=dict(color='green'),
            yaxis='y1'
        ))

        colors = ['red', 'orange', 'purple', 'gray']
        for i, period in enumerate(selected_smas):
            fig.add_trace(go.Scatter(
                x=self.data['date'],
                y=self.data[f'SMA{period}'],
                mode='lines',
                name=f'SMA{period}',
                line=dict(color=colors[i % len(colors)], width=2),
                yaxis='y1'
            ))

        fig.update_layout(
            title=f"{self.asset_id.upper()} Price and SMAs",
            xaxis_title="Date",
            yaxis_title=f"Price ({self.price_column})",
            yaxis=dict(type='log', autorange=True, gridcolor='lightgray'),
            xaxis=dict(gridcolor='lightgray'),
            template="plotly_white",
            showlegend=True,
            height=500,
            margin=dict(l=50, r=50, t=100, b=50)
        )
        return fig

    def visualize(self):
        self.create_table()
        chart_fig = self.create_chart()
        chart_fig.show()

# Usage
analyzer = TrendAnalyzer(
    asset_id='render-token',
    data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/',
    btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv',
    use_btc_adjusted=False
)
analyzer.visualize()



Initial columns after loading asset data: ['date', 'open', 'high', 'low', 'close']
Bitcoin data columns: ['date', 'btc_close']
Columns after merge: ['date', 'open', 'high', 'low', 'close', 'btc_close']
Columns after dropping 'btc_close': ['date', 'open', 'high', 'low', 'close', 'render-token_btc']
Price column selected: close
Columns after numeric conversion and dropna: ['date', 'open', 'high', 'low', 'close', 'render-token_btc']
Columns after calculating indicators: ['date', 'open', 'high', 'low', 'close', 'render-token_btc', 'SMA3', 'SMA5', 'SMA7', 'SMA14', 'SMA21', 'SMA30', 'SMA45', 'SMA63', 'SMA84', 'SMA100', 'SMA120', 'SMA150', 'SMA200', 'SMA252', 'SMA365', 'RoC3', 'RoC5', 'RoC7', 'RoC14', 'RoC21', 'RoC30', 'RoC45', 'RoC63', 'RoC84', 'RoC100', 'RoC120', 'RoC150', 'RoC200', 'RoC252', 'RoC365']
Short Term classification: Strong Bear
Medium Term classification: Strong Bear
Long Term classification: Strong Bear
Overall classification: Strong Bear
Table data: {'Ticker': ['RENDER-TOKEN'

In [127]:
import plotly.express as px

# Define the trend order and color mapping
trend_order = ['Strong Bull', 'Weak Bull', 'Neutral', 'Weak Bear', 'Strong Bear']
color_map = {
    'Strong Bull': 'darkgreen',
    'Weak Bull': 'lightgreen',
    'Neutral': 'gray',
    'Weak Bear': 'orange',
    'Strong Bear': 'red'
}

# Create a scatter plot of Overall trend classification through time using plotly
fig = px.scatter(analyzer.classified_data, x='date', y='Overall', 
                 title='Overall Trend Classification Through Time',
                 color='Overall',
                 color_discrete_map=color_map,
                 category_orders={'Overall': trend_order})

fig.update_layout(
    xaxis_title='',
    yaxis_title='Trend Classification',
    template='plotly_white',
    height=600,
    width=1000,
    yaxis=dict(
        categoryorder='array',
        categoryarray=trend_order
    )
)

fig.update_traces(marker=dict(size=4))
fig.show()

In [128]:
# Define a simple trading strategy based on trend classification
def trend_based_strategy(data):
    """
    Trading strategy that:
    - Goes long at market open if previous day's close was classified as 'weak bull' or 'strong bull'
    - Stays in cash otherwise
    
    Args:
        data: DataFrame with trend classifications and price data
    
    Returns:
        DataFrame with strategy signals and performance metrics
    """
    # Create a copy of the data to avoid modifying the original
    strategy_data = data.copy()
    
    # Create a signal column (1 for long, 0 for cash)
    strategy_data['signal'] = 0
    
    # Generate signals based on previous day's classification
    bullish_conditions = (strategy_data['Overall'].shift(1).isin(['Weak Bull', 'Strong Bull']))
    strategy_data.loc[bullish_conditions, 'signal'] = 1
    # Calculate returns
    strategy_data['daily_return'] = strategy_data['close'].pct_change()
    
    # Calculate strategy returns (signal from previous day * today's return)
    strategy_data['strategy_return'] = strategy_data['signal'] * strategy_data['daily_return']
    
    # Calculate cumulative returns
    strategy_data['cumulative_return'] = (1 + strategy_data['daily_return']).cumprod() - 1
    strategy_data['strategy_cumulative_return'] = (1 + strategy_data['strategy_return']).cumprod() - 1
    
    return strategy_data

# Apply the strategy to our classified data
strategy_results = trend_based_strategy(analyzer.classified_data)

# Visualize the strategy performance
fig = px.line(strategy_results, x='date', y=['cumulative_return', 'strategy_cumulative_return'],
              title='Trend-Based Trading Strategy Performance',
              labels={'value': 'Cumulative Return', 'variable': 'Strategy'},
              color_discrete_map={
                  'cumulative_return': 'gray',
                  'strategy_cumulative_return': 'blue'
              })

fig.update_layout(
    xaxis_title='',
    yaxis_title='Cumulative Return',
    template='plotly_white',
    height=600,
    width=1000,
    legend=dict(
        title=None,
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

# Update legend labels
fig.for_each_trace(lambda t: t.update(name='Buy & Hold' if t.name == 'cumulative_return' else 'Trend Strategy'))

fig.show()

# Calculate performance metrics
total_days = len(strategy_results)
invested_days = strategy_results['signal'].sum()
percent_invested = invested_days / total_days * 100

# Calculate annualized returns
days_in_market = (strategy_results['date'].iloc[-1] - strategy_results['date'].iloc[0]).days
annual_factor = 365 / days_in_market
buy_hold_return = strategy_results['cumulative_return'].iloc[-1]
strategy_return = strategy_results['strategy_cumulative_return'].iloc[-1]
annualized_buy_hold = (1 + buy_hold_return) ** annual_factor - 1
annualized_strategy = (1 + strategy_return) ** annual_factor - 1

# Print performance summary
print(f"Strategy Performance Summary:")
print(f"Period: {strategy_results['date'].iloc[0].date()} to {strategy_results['date'].iloc[-1].date()} ({days_in_market} days)")
print(f"Time invested in market: {percent_invested:.2f}%")
print(f"Buy & Hold return: {buy_hold_return:.2%} (Annualized: {annualized_buy_hold:.2%})")
print(f"Strategy return: {strategy_return:.2%} (Annualized: {annualized_strategy:.2%})")
print(f"Outperformance: {strategy_return - buy_hold_return:.2%}")


Strategy Performance Summary:
Period: 2020-06-16 to 2025-03-09 (1727 days)
Time invested in market: 25.00%
Buy & Hold return: 6718.00% (Annualized: 144.09%)
Strategy return: 9367.86% (Annualized: 161.62%)
Outperformance: 2649.86%
